# Household Power Consumption Dataset Inspection

This notebook loads the household electricity consumption data for a first review before any cleaning or modeling.


In [1]:
from pathlib import Path

import pandas as pd

data_path = Path("../data/household_power_consumption.txt")
if not data_path.exists():
    data_path = Path("data/household_power_consumption.txt")

# Load the semicolon-separated file and treat ? as missing values.
df = pd.read_csv(data_path, sep=';', na_values='?')

df.head()


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
0,16/12/2006,17:24:00,4.216,0.418,234.84,18.4,0.0,1.0,17.0
1,16/12/2006,17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0
2,16/12/2006,17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0
3,16/12/2006,17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0
4,16/12/2006,17:28:00,3.666,0.528,235.68,15.8,0.0,1.0,17.0


### 1. Inspect the first rows, shape, columns, and dtypes

Display the first rows and basic dataset structure.


In [2]:
print("First 5 rows:")
display(df.head())
print("\nRows:", df.shape[0])
print("Columns:", df.shape[1])
print("\nColumn names:")
print(df.columns.tolist())
print("\nData types:")
display(df.dtypes)


First 5 rows:


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
0,16/12/2006,17:24:00,4.216,0.418,234.84,18.4,0.0,1.0,17.0
1,16/12/2006,17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0
2,16/12/2006,17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0
3,16/12/2006,17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0
4,16/12/2006,17:28:00,3.666,0.528,235.68,15.8,0.0,1.0,17.0



Rows: 2075259
Columns: 9

Column names:
['Date', 'Time', 'Global_active_power', 'Global_reactive_power', 'Voltage', 'Global_intensity', 'Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']

Data types:


Date                         str
Time                         str
Global_active_power      float64
Global_reactive_power    float64
Voltage                  float64
Global_intensity         float64
Sub_metering_1           float64
Sub_metering_2           float64
Sub_metering_3           float64
dtype: object

### 2. Create a timestamp from Date and Time

Keep the original columns and add a combined timestamp for inspection.


In [3]:
df["timestamp"] = pd.to_datetime(df["Date"] + " " + df["Time"])
print(df[["Date", "Time", "timestamp"]].head())


         Date      Time           timestamp
0  16/12/2006  17:24:00 2006-12-16 17:24:00
1  16/12/2006  17:25:00 2006-12-16 17:25:00
2  16/12/2006  17:26:00 2006-12-16 17:26:00
3  16/12/2006  17:27:00 2006-12-16 17:27:00
4  16/12/2006  17:28:00 2006-12-16 17:28:00


/var/folders/t1/pd3xbl0s3mv037q4k1zws1n40000gn/T/ipykernel_19810/2500894058.py:1: UserWarning: Parsing dates in %d/%m/%Y %H:%M:%S format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df["timestamp"] = pd.to_datetime(df["Date"] + " " + df["Time"])


### 3. Check missing values by column

Count missing values for each column before any cleaning work.


In [4]:
missing_values = df.isnull().sum()
print("Missing values by column:")
display(missing_values)


Missing values by column:


Date                         0
Time                         0
Global_active_power      25979
Global_reactive_power    25979
Voltage                  25979
Global_intensity         25979
Sub_metering_1           25979
Sub_metering_2           25979
Sub_metering_3           25979
timestamp                    0
dtype: int64

### 4. Check time gaps between consecutive timestamps

Compare each timestamp with the next one to see whether the time series is regular.


In [5]:
df["time_gap_minutes"] = df["timestamp"].diff().dt.total_seconds() / 60
print("First few time gaps in minutes:")
display(df[["timestamp", "time_gap_minutes"]].head())
print("\nSummary of time gaps:")
display(df["time_gap_minutes"].describe())


First few time gaps in minutes:


,timestamp,time_gap_minutes
0,2006-12-16 17:24:00,NaN
1,2006-12-16 17:25:00,1.0
2,2006-12-16 17:26:00,1.0
3,2006-12-16 17:27:00,1.0
4,2006-12-16 17:28:00,1.0



Summary of time gaps:


count    2075258.0
mean           1.0
std            0.0
min            1.0
25%            1.0
50%            1.0
75%            1.0
max            1.0
Name: time_gap_minutes, dtype: float64

### Findings summary

- The file was loaded successfully as a semicolon-separated dataset with missing values represented by `?`.
- The combined timestamp column gives a clear time index for inspection without removing the original Date and Time columns.
- Missing values are present in some columns and should be reviewed before any modeling.
- The time-gap check shows whether the data is recorded at regular one-minute intervals or whether there are irregular gaps.
